# Mechanistic digital twin for cancer-cell migration under interstitial flow

This notebook implements the final, frozen proof-of-concept model developed from the exploratory notebook. It contains the complete reproducible workflow:

1. flow-dependent amoeboid-mesenchymal phenotype switching;
2. stochastic phenotype-dependent migration;
3. synthetic microscopy measurements;
4. prediction-measurement-state update with a particle filter;
5. reverse inference of latent phenotype;
6. withheld-data trajectory forecasting; and
7. independent validation against hold-position and linear-extrapolation baselines.

**Scope.** Published experimental summary statistics from Huang *et al.* (2015) are used as calibration targets. Trajectory-level state estimation and forecasting are evaluated on held-out **synthetic** data. This notebook does not claim validation on the original experimental tracks.

**Reproducibility.** All seeds and final parameters are fixed. Run with **Kernel > Restart & Run All**. Figures and CSV tables are written to `outputs/`.

## 1 Imports and configuration

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

@dataclass(frozen=True)
class ModelConfig:
    # Numerical time is in model units. The complete run maps to 16 experimental hours.
    dt: float = 0.05
    n_steps: int = 1000
    experiment_hours: float = 16.0
    n_frames: int = 192
    initial_mesenchymal_fraction: float = 0.25
    phenotype_relaxation_hours: float = 4.0

    # Frozen migration parameters from the calibration search.
    active_speed: float = 1.0
    positional_diffusion: float = 0.25
    tau_amoeboid: float = 0.50       # model-time units
    tau_mesenchymal: float = 12.0    # model-time units
    speed_multiplier_amoeboid: float = 2.20
    speed_multiplier_mesenchymal: float = 1.00

    # Observation and particle-filter parameters.
    measurement_std: float = 0.50
    likelihood_scale: float = 1.25
    resampling_fraction: float = 0.50
    assimilation_cutoff_frame: int = 96

CFG = ModelConfig()
HOURS_PER_MODEL_UNIT = CFG.experiment_hours / (CFG.n_steps * CFG.dt)
DT_HOURS = CFG.dt * HOURS_PER_MODEL_UNIT
SAMPLE_INDICES = np.round(np.linspace(0, CFG.n_steps, CFG.n_frames)).astype(int)
FRAME_HOURS = np.arange(CFG.n_frames) * (5.0 / 60.0)
FORECAST_HORIZONS_HOURS = np.array([0.5, 1.0, 2.0, 4.0, 8.0])

print(CFG)
print("Fine time step in experimental hours:", DT_HOURS)
print("Experimental sample frames:", len(SAMPLE_INDICES))

## 2 Experimental targets and phenotype-switching rates

The published 16-hour mesenchymal fractions are 0.58 without flow and 0.27 with flow, starting from a fraction of 0.25. A two-state continuous-time Markov model uses

\[
\frac{df_M}{dt}=k_{A\to M}(1-f_M)-k_{M\to A}f_M.
\]

The relaxation time is fixed at 4 h. The rates below reproduce the target final fractions in the corresponding mean-field model.

In [ ]:
TARGETS = {
    "persistence_no_flow": 0.260,
    "persistence_flow": 0.160,
    "alpha_no_flow": 1.460,
    "alpha_flow": 1.270,
    "speed_ratio": 1.090,
    "mesenchymal_fraction_no_flow": 0.580,
    "mesenchymal_fraction_flow": 0.270,
}

def equilibrium_fraction(final_fraction, initial_fraction, duration, relaxation_time):
    decay = np.exp(-duration / relaxation_time)
    return (final_fraction - initial_fraction * decay) / (1.0 - decay)

def transition_rates(equilibrium_mesenchymal_fraction, relaxation_time):
    return (
        equilibrium_mesenchymal_fraction / relaxation_time,
        (1.0 - equilibrium_mesenchymal_fraction) / relaxation_time,
    )

equilibrium_no_flow = equilibrium_fraction(
    TARGETS["mesenchymal_fraction_no_flow"], CFG.initial_mesenchymal_fraction,
    CFG.experiment_hours, CFG.phenotype_relaxation_hours,
)
equilibrium_flow = equilibrium_fraction(
    TARGETS["mesenchymal_fraction_flow"], CFG.initial_mesenchymal_fraction,
    CFG.experiment_hours, CFG.phenotype_relaxation_hours,
)
RATES = {
    False: transition_rates(equilibrium_no_flow, CFG.phenotype_relaxation_hours),
    True: transition_rates(equilibrium_flow, CFG.phenotype_relaxation_hours),
}

print("No-flow equilibrium fraction:", equilibrium_no_flow)
print("Flow equilibrium fraction:", equilibrium_flow)
print("No-flow rates A->M, M->A:", RATES[False])
print("Flow rates A->M, M->A:", RATES[True])

## 3 Mechanistic phenotype-switching migration model

In [ ]:
def simulate_population(seed, flow_condition, number_of_cells=192):
    """Simulate hidden position and phenotype histories."""
    rng = np.random.default_rng(seed)
    rate_a_to_m, rate_m_to_a = RATES[bool(flow_condition)]
    p_a_to_m = 1.0 - np.exp(-rate_a_to_m * DT_HOURS)
    p_m_to_a = 1.0 - np.exp(-rate_m_to_a * DT_HOURS)

    phenotype = rng.random(number_of_cells) < CFG.initial_mesenchymal_fraction
    positions = np.zeros((number_of_cells, 2))
    velocities = np.zeros((number_of_cells, 2))
    trajectories = np.zeros((CFG.n_steps + 1, number_of_cells, 2))
    phenotype_history = np.zeros((CFG.n_steps + 1, number_of_cells), dtype=bool)
    phenotype_history[0] = phenotype

    for step in range(CFG.n_steps):
        u = rng.random(number_of_cells)
        switch_a_to_m = (~phenotype) & (u < p_a_to_m)
        switch_m_to_a = phenotype & (u < p_m_to_a)
        phenotype = phenotype.copy()
        phenotype[switch_a_to_m] = True
        phenotype[switch_m_to_a] = False

        tau = np.where(phenotype, CFG.tau_mesenchymal, CFG.tau_amoeboid)
        speed = np.where(
            phenotype,
            CFG.active_speed * CFG.speed_multiplier_mesenchymal,
            CFG.active_speed * CFG.speed_multiplier_amoeboid,
        )
        velocities += -(velocities / tau[:, None]) * CFG.dt
        velocities += speed[:, None] * np.sqrt(2 * CFG.dt / tau[:, None]) * rng.normal(size=velocities.shape)
        positions += velocities * CFG.dt
        positions += np.sqrt(2 * CFG.positional_diffusion * CFG.dt) * rng.normal(size=positions.shape)
        trajectories[step + 1] = positions
        phenotype_history[step + 1] = phenotype

    return trajectories, phenotype_history

def calculate_persistence(sampled_tracks):
    net = np.linalg.norm(sampled_tracks[-1] - sampled_tracks[0], axis=1)
    path = np.linalg.norm(np.diff(sampled_tracks, axis=0), axis=2).sum(axis=0)
    return np.divide(net, path, out=np.zeros_like(net), where=path > 0)

def calculate_speed(sampled_tracks):
    path = np.linalg.norm(np.diff(sampled_tracks, axis=0), axis=2).sum(axis=0)
    return path / CFG.experiment_hours

def time_averaged_msd(sampled_tracks):
    n = sampled_tracks.shape[0]
    msd = np.zeros(n)
    for lag in range(1, n):
        dr = sampled_tracks[lag:] - sampled_tracks[:-lag]
        msd[lag] = np.mean(np.sum(dr**2, axis=2))
    return msd

def msd_exponent(sampled_tracks):
    lag_minutes = np.arange(sampled_tracks.shape[0]) * 5.0
    msd = time_averaged_msd(sampled_tracks)
    mask = (lag_minutes >= 5.0) & (msd > 0)
    return np.polyfit(np.log(lag_minutes[mask]), np.log(msd[mask]), 1)[0]

## 4 Frozen-model calibration summary

The parameter search was performed in the exploratory notebook and is intentionally not repeated here. The table records the independent 10-seed validation of the frozen parameters. Six of seven means are within 4% of the published target; persistence under flow is the main residual mismatch.

In [ ]:
calibration_summary = pd.DataFrame({
    "observable": ["Persistence no flow", "Persistence flow", "MSD exponent no flow", "MSD exponent flow", "Speed ratio", "Mesenchymal fraction no flow", "Mesenchymal fraction flow"],
    "simulation_mean": [0.252, 0.193, 1.442, 1.317, 1.065, 0.591, 0.272],
    "simulation_sd": [0.008, 0.011, 0.019, 0.037, 0.013, 0.034, 0.024],
    "experimental_target": [0.260, 0.160, 1.460, 1.270, 1.090, 0.580, 0.270],
})
calibration_summary["relative_error_percent"] = 100 * (
    calibration_summary.simulation_mean - calibration_summary.experimental_target
) / calibration_summary.experimental_target
calibration_summary.to_csv(OUTPUT_DIR / "calibration_summary.csv", index=False)
calibration_summary.round(3)

## 5 Generate an independent synthetic flow experiment

In [ ]:
true_tracks, true_phenotypes = simulate_population(seed=80000, flow_condition=True, number_of_cells=20)
true_sampled = true_tracks[SAMPLE_INDICES]
phenotypes_sampled = true_phenotypes[SAMPLE_INDICES]
measurement_rng = np.random.default_rng(81000)
measurements = true_sampled + measurement_rng.normal(0.0, CFG.measurement_std, true_sampled.shape)

print("Hidden trajectory shape:", true_sampled.shape)
print("Measurement shape:", measurements.shape)
print("Initial/final mesenchymal fraction:", phenotypes_sampled[0].mean(), phenotypes_sampled[-1].mean())

## 6 Particle-filter digital twin

Each particle contains position, velocity, and a binary phenotype. Between frames the mechanistic model propagates all particles. A Gaussian position likelihood updates their weights. The weighted mesenchymal fraction is the reverse prediction, \(P(M\mid y_{1:k})\). Systematic resampling is triggered when the effective sample size is less than half the ensemble.

In [ ]:
def systematic_resample(weights, rng):
    n = len(weights)
    points = rng.random() / n + np.arange(n) / n
    return np.searchsorted(np.cumsum(weights), points, side="right")

def propagate_particles(positions, velocities, phenotypes, fine_steps, rng, flow_condition=True):
    positions, velocities, phenotypes = positions.copy(), velocities.copy(), phenotypes.copy()
    rate_a_to_m, rate_m_to_a = RATES[bool(flow_condition)]
    p_a_to_m = 1.0 - np.exp(-rate_a_to_m * DT_HOURS)
    p_m_to_a = 1.0 - np.exp(-rate_m_to_a * DT_HOURS)

    for _ in range(fine_steps):
        u = rng.random(len(phenotypes))
        a_to_m = (~phenotypes) & (u < p_a_to_m)
        m_to_a = phenotypes & (u < p_m_to_a)
        phenotypes[a_to_m] = True
        phenotypes[m_to_a] = False
        tau = np.where(phenotypes, CFG.tau_mesenchymal, CFG.tau_amoeboid)
        speed = np.where(
            phenotypes,
            CFG.active_speed * CFG.speed_multiplier_mesenchymal,
            CFG.active_speed * CFG.speed_multiplier_amoeboid,
        )
        velocities += -(velocities / tau[:, None]) * CFG.dt
        velocities += speed[:, None] * np.sqrt(2 * CFG.dt / tau[:, None]) * rng.normal(size=velocities.shape)
        positions += velocities * CFG.dt
        positions += np.sqrt(2 * CFG.positional_diffusion * CFG.dt) * rng.normal(size=positions.shape)
    return positions, velocities, phenotypes

def run_particle_filter(cell_index, true_sampled, phenotype_sampled, measurements,
                        number_of_particles=1000, random_seed=84000):
    rng = np.random.default_rng(random_seed + cell_index)
    truth, observed = true_sampled[:, cell_index], measurements[:, cell_index]
    labels = phenotype_sampled[:, cell_index].astype(int)
    n = number_of_particles
    positions = observed[0] + rng.normal(0, CFG.measurement_std, (n, 2))
    velocities = rng.normal(0, CFG.active_speed, (n, 2))
    phenotypes = rng.random(n) < CFG.initial_mesenchymal_fraction
    weights = np.full(n, 1.0 / n)
    predicted = np.full((CFG.n_frames, 2), np.nan)
    updated = np.full((CFG.n_frames, 2), np.nan)
    phenotype_probability = np.full(CFG.n_frames, np.nan)
    effective_sample_size = np.full(CFG.n_frames, np.nan)
    updated[0] = np.average(positions, axis=0, weights=weights)
    phenotype_probability[0] = np.average(phenotypes, weights=weights)
    effective_sample_size[0] = n
    resampled = np.zeros(CFG.n_frames, dtype=bool)
    likelihood_std = CFG.likelihood_scale * CFG.measurement_std

    for frame in range(1, CFG.n_frames):
        fine_steps = SAMPLE_INDICES[frame] - SAMPLE_INDICES[frame - 1]
        p_pos, p_vel, p_pheno = propagate_particles(positions, velocities, phenotypes, fine_steps, rng, True)
        predicted[frame] = np.average(p_pos, axis=0, weights=weights)
        residual = observed[frame] - p_pos
        log_w = np.log(weights + 1e-300) - 0.5 * np.sum(residual**2, axis=1) / likelihood_std**2
        log_w -= log_w.max()
        new_weights = np.exp(log_w)
        new_weights /= new_weights.sum()
        updated[frame] = np.average(p_pos, axis=0, weights=new_weights)
        phenotype_probability[frame] = np.average(p_pheno, weights=new_weights)
        ess = 1.0 / np.sum(new_weights**2)
        effective_sample_size[frame] = ess

        if ess < CFG.resampling_fraction * n:
            idx = systematic_resample(new_weights, rng)
            positions, velocities, phenotypes = p_pos[idx].copy(), p_vel[idx].copy(), p_pheno[idx].copy()
            weights = np.full(n, 1.0 / n)
            resampled[frame] = True
        else:
            positions, velocities, phenotypes, weights = p_pos, p_vel, p_pheno, new_weights

    prediction_rmse = np.sqrt(np.nanmean(np.sum((predicted[1:] - truth[1:])**2, axis=1)))
    measurement_rmse = np.sqrt(np.mean(np.sum((observed - truth)**2, axis=1)))
    updated_rmse = np.sqrt(np.mean(np.sum((updated - truth)**2, axis=1)))
    predicted_labels = (phenotype_probability >= 0.5).astype(int)
    auc = roc_auc_score(labels, phenotype_probability) if len(np.unique(labels)) > 1 else np.nan
    balanced = balanced_accuracy_score(labels, predicted_labels) if len(np.unique(labels)) > 1 else np.nan
    return {
        "cell": cell_index + 1,
        "prediction_rmse": prediction_rmse,
        "measurement_rmse": measurement_rmse,
        "updated_rmse": updated_rmse,
        "frames_better": int(np.sum(np.linalg.norm(updated-truth, axis=1) < np.linalg.norm(observed-truth, axis=1))),
        "phenotype_auc": auc,
        "balanced_accuracy": balanced,
        "phenotype_brier": np.mean((phenotype_probability-labels)**2),
        "resampling_percent": 100 * resampled.sum() / (CFG.n_frames - 1),
        "predicted_history": predicted,
        "updated_history": updated,
        "phenotype_probability": phenotype_probability,
        "ess": effective_sample_size,
    }

## 7 State reconstruction and reverse phenotype inference

In [ ]:
filter_results = [
    run_particle_filter(i, true_sampled, phenotypes_sampled, measurements)
    for i in range(20)
]

prediction_rmse = np.sqrt(np.mean([r["prediction_rmse"]**2 for r in filter_results]))
measurement_rmse = np.sqrt(np.mean([r["measurement_rmse"]**2 for r in filter_results]))
updated_rmse = np.sqrt(np.mean([r["updated_rmse"]**2 for r in filter_results]))
mean_auc = np.nanmean([r["phenotype_auc"] for r in filter_results])
mean_balanced = np.nanmean([r["balanced_accuracy"] for r in filter_results])
mean_brier = np.mean([r["phenotype_brier"] for r in filter_results])
frames_better = sum(r["frames_better"] for r in filter_results)

assimilation_summary = pd.DataFrame({
    "metric": ["prediction_rmse", "measurement_rmse", "updated_rmse", "rmse_improvement_percent", "frames_better", "total_frames", "mean_auc", "mean_balanced_accuracy", "mean_brier"],
    "value": [prediction_rmse, measurement_rmse, updated_rmse, 100*(measurement_rmse-updated_rmse)/measurement_rmse, frames_better, 20*CFG.n_frames, mean_auc, mean_balanced, mean_brier],
})
assimilation_summary.to_csv(OUTPUT_DIR / "assimilation_summary.csv", index=False)
assimilation_summary

In [ ]:
# Representative forward and reverse reconstruction.
i = 0
r = filter_results[i]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(true_sampled[:, i, 0], true_sampled[:, i, 1], color="black", lw=2, label="Hidden truth")
axes[0].scatter(measurements[:, i, 0], measurements[:, i, 1], s=8, alpha=.35, label="Measurements")
axes[0].plot(r["updated_history"][:, 0], r["updated_history"][:, 1], color="tab:blue", lw=1.5, label="Updated twin")
axes[0].set(xlabel="x", ylabel="y", title="A. State reconstruction")
axes[0].legend(frameon=False)
axes[1].plot(FRAME_HOURS, phenotypes_sampled[:, i].astype(float), color="black", lw=1.2, drawstyle="steps-post", label="Hidden state")
axes[1].plot(FRAME_HOURS, r["phenotype_probability"], color="tab:purple", lw=2, label="Posterior P(mesenchymal)")
axes[1].axhline(.5, color="gray", ls="--", lw=1)
axes[1].set(xlabel="Time (h)", ylabel="Probability", ylim=(-.05,1.05), title="B. Reverse phenotype inference")
axes[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "state_and_phenotype_inference.png", dpi=300, bbox_inches="tight")
plt.show()

## 8 Withheld-data forecasting

In [ ]:
def forecast_one_cell(cell_index, true_sampled, phenotype_sampled, measurements,
                      number_of_particles=300, random_seed=89000):
    rng = np.random.default_rng(random_seed + cell_index)
    truth, observed = true_sampled[:, cell_index], measurements[:, cell_index]
    n = number_of_particles
    positions = observed[0] + rng.normal(0, CFG.measurement_std, (n,2))
    velocities = rng.normal(0, CFG.active_speed, (n,2))
    phenotypes = rng.random(n) < CFG.initial_mesenchymal_fraction
    weights = np.full(n, 1/n)
    likelihood_std = CFG.likelihood_scale * CFG.measurement_std

    # Assimilate first 8 hours.
    for frame in range(1, CFG.assimilation_cutoff_frame + 1):
        fine_steps = SAMPLE_INDICES[frame] - SAMPLE_INDICES[frame-1]
        p_pos, p_vel, p_pheno = propagate_particles(positions, velocities, phenotypes, fine_steps, rng, True)
        residual = observed[frame] - p_pos
        log_w = np.log(weights + 1e-300) - .5*np.sum(residual**2,axis=1)/likelihood_std**2
        log_w -= log_w.max(); new_w = np.exp(log_w); new_w /= new_w.sum()
        if 1/np.sum(new_w**2) < CFG.resampling_fraction*n:
            idx = systematic_resample(new_w, rng)
            positions, velocities, phenotypes = p_pos[idx].copy(), p_vel[idx].copy(), p_pheno[idx].copy()
            weights = np.full(n, 1/n)
        else:
            positions, velocities, phenotypes, weights = p_pos, p_vel, p_pheno, new_w

    idx = systematic_resample(weights, rng)
    positions, velocities, phenotypes = positions[idx].copy(), velocities[idx].copy(), phenotypes[idx].copy()
    mean = np.full((CFG.n_frames,2), np.nan)
    lower = np.full_like(mean, np.nan); upper = np.full_like(mean, np.nan)
    cutoff = CFG.assimilation_cutoff_frame
    mean[cutoff] = positions.mean(axis=0)
    lower[cutoff], upper[cutoff] = np.quantile(positions,[.025,.975],axis=0)

    for frame in range(cutoff+1, CFG.n_frames):
        fine_steps = SAMPLE_INDICES[frame] - SAMPLE_INDICES[frame-1]
        positions, velocities, phenotypes = propagate_particles(positions, velocities, phenotypes, fine_steps, rng, True)
        mean[frame] = positions.mean(axis=0)
        lower[frame], upper[frame] = np.quantile(positions,[.025,.975],axis=0)

    twin_rmse=[]; hold_rmse=[]; linear_rmse=[]; coverage=[]; width=[]
    hist_idx=np.arange(cutoff-12, cutoff+1); hist_t=(hist_idx-cutoff)*5/60
    bx=np.polyfit(hist_t, observed[hist_idx,0],1); by=np.polyfit(hist_t, observed[hist_idx,1],1)
    velocity=np.array([bx[0],by[0]]); fitted_cutoff=np.array([bx[1],by[1]])
    for hours in FORECAST_HORIZONS_HOURS:
        end=min(cutoff+int(round(hours*60/5)),CFG.n_frames-1)
        sl=slice(cutoff+1,end+1); target=truth[sl]; pred=mean[sl]
        hold=np.repeat(observed[cutoff][None,:],len(target),axis=0)
        future_t=np.arange(1,len(target)+1)*5/60
        linear=fitted_cutoff+future_t[:,None]*velocity
        rmse=lambda a: np.sqrt(np.mean(np.sum((a-target)**2,axis=1)))
        twin_rmse.append(rmse(pred)); hold_rmse.append(rmse(hold)); linear_rmse.append(rmse(linear))
        coverage.append(np.mean((target>=lower[sl])&(target<=upper[sl])))
        width.append(np.mean(upper[sl]-lower[sl]))
    return {"twin_rmse":np.array(twin_rmse),"hold_rmse":np.array(hold_rmse),"linear_rmse":np.array(linear_rmse),"coverage":np.array(coverage),"width":np.array(width)}

## 9 Independent forecast validation

Three new synthetic flow experiments (20 cells each) are generated after the model is frozen. The twin assimilates the first 8 h and forecasts the remaining interval without additional measurements.

In [ ]:
validation_results=[]
for experiment in range(3):
    tracks, states = simulate_population(87000+experiment, True, 20)
    truth = tracks[SAMPLE_INDICES]; labels = states[SAMPLE_INDICES]
    rng = np.random.default_rng(88000+experiment)
    observed = truth + rng.normal(0, CFG.measurement_std, truth.shape)
    for cell in range(20):
        validation_results.append(forecast_one_cell(cell,truth,labels,observed,300,89000+100*experiment))
    print(f"Completed independent experiment {experiment+1}/3")

def root_mean_square_across_cells(key):
    values=np.stack([r[key] for r in validation_results])
    return np.sqrt(np.mean(values**2,axis=0))

twin_rmse=root_mean_square_across_cells("twin_rmse")
hold_rmse=root_mean_square_across_cells("hold_rmse")
linear_rmse=root_mean_square_across_cells("linear_rmse")
coverage=np.mean(np.stack([r["coverage"] for r in validation_results]),axis=0)
width=np.mean(np.stack([r["width"] for r in validation_results]),axis=0)
forecast_summary=pd.DataFrame({
    "horizon_hours":FORECAST_HORIZONS_HOURS,
    "twin_rmse":twin_rmse,"hold_rmse":hold_rmse,"linear_rmse":linear_rmse,
    "improvement_vs_hold_percent":100*(hold_rmse-twin_rmse)/hold_rmse,
    "improvement_vs_linear_percent":100*(linear_rmse-twin_rmse)/linear_rmse,
    "coverage":coverage,"mean_interval_width":width,
})
forecast_summary.to_csv(OUTPUT_DIR/"independent_forecast_summary.csv",index=False)
forecast_summary.round(3)

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,4))
axes[0].plot(FORECAST_HORIZONS_HOURS,twin_rmse,"o-",label="Digital twin")
axes[0].plot(FORECAST_HORIZONS_HOURS,hold_rmse,"s--",label="Hold position")
axes[0].plot(FORECAST_HORIZONS_HOURS,linear_rmse,"^--",label="Linear extrapolation")
axes[0].set(xlabel="Forecast horizon (h)",ylabel="Position RMSE",title="A. Forecast error"); axes[0].legend(frameon=False); axes[0].grid(alpha=.2)
axes[1].bar(FORECAST_HORIZONS_HOURS.astype(str),100*(linear_rmse-twin_rmse)/linear_rmse)
axes[1].set(xlabel="Forecast horizon (h)",ylabel="RMSE reduction (%)",title="B. Improvement over linear"); axes[1].grid(axis="y",alpha=.2)
axes[2].plot(FORECAST_HORIZONS_HOURS,coverage,"o-",color="tab:green"); axes[2].axhline(.95,color="black",ls="--",label="Nominal 95%")
axes[2].set(xlabel="Forecast horizon (h)",ylabel="Coordinate coverage",ylim=(.85,1),title="C. Uncertainty calibration"); axes[2].legend(frameon=False); axes[2].grid(alpha=.2)
fig.tight_layout(); fig.savefig(OUTPUT_DIR/"independent_forecast_validation.png",dpi=300,bbox_inches="tight"); plt.show()

## 10 Interpretation

- The updated particle-filter state should be compared with noisy measurements to quantify assimilation benefit.
- The phenotype probability is a **model-based latent-state estimate**, not an experimentally measured label.
- The strongest forecast claim is the comparison with linear extrapolation. Performance against the hold-position baseline is much closer and can be neutral or slightly worse at the longest horizon.
- Near-95% empirical coverage demonstrates calibration only for these synthetic held-out experiments.
- Prospective validation requires experimental tracks with independently assigned phenotype labels.

In [ ]:
run_metadata = {
    "model_config": CFG.__dict__,
    "seeds": {"heldout_truth":80000,"heldout_measurement":81000,"filter":84000,"validation_truth_start":87000,"validation_measurement_start":88000,"validation_filter_start":89000},
    "validation_cells":60,
    "calibration_source":"Published aggregate observables; Huang et al. (2015)",
    "trajectory_validation":"Synthetic held-out trajectories",
}
(OUTPUT_DIR/"run_metadata.json").write_text(json.dumps(run_metadata,indent=2))
print("Saved reproducibility outputs to",OUTPUT_DIR.resolve())